In [ ]:
import json

from langchain_core.documents import Document
from langchain.embeddings import Embeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

from fastembed import TextEmbedding

In [29]:
KNOWLEDGE_FILE = "../data/processed/fil_ag.jsonl"

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "math_notes"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

In [30]:
# FastEmbed -> LangChain adapter

class FastEmbeddingEmbed(Embeddings):
    def __init__(self, model_name: str):
        self.model = TextEmbedding(model_name=model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        vectors = self.model.embed(texts)
        return [vector.tolist() for vector in vectors]

    def embed_query(self, text: str) -> list[float]:
        return self.embed_documents([text])[0]

In [31]:
def element_to_text(element: dict) -> str:
    element_type = element["type"]

    if element_type == "text":
        return element["text"]

    if element_type == "formula":
        return f"FORMULA:\n{element['text']}"

    if element_type == "image":
        return f"IMAGE: {element['path']}"

    return str(element)


def theorem_to_text(item: dict) -> str:
    parts = []

    parts.append(
        f"Тип знания: теорема"
    )

    parts.append(
        f"Глава: {item.get('chapter', '')}"
    )

    parts.append(
        f"Тема: {item.get('chapter_name', '')}"
    )

    parts.append(
        f"Раздел: {item.get('section', '')}"
    )

    statement = []

    for element in item.get("statement", []):
        statement.append(
            element_to_text(element)
        )

    if statement:
        parts.append(
            "Формулировка теоремы:\n"
            + "\n".join(statement)
        )

    proof = []

    for element in item.get("proof", []):
        proof.append(
            element_to_text(element)
        )

    if proof:
        parts.append(
            "Доказательство:\n"
            + "\n".join(proof)
        )

    return "\n\n".join(parts)


def definition_to_text(item: dict) -> str:
    parts = []

    parts.append(
        "Тип знания: определение"
    )

    parts.append(
        f"Глава: {item.get('chapter', '')}"
    )

    parts.append(
        f"Тема: {item.get('chapter_name', '')}"
    )

    parts.append(
        f"Раздел: {item.get('section', '')}"
    )

    content = []

    for element in item.get("content", []):
        content.append(
            element_to_text(element)
        )

    if content:
        parts.append(
            "Содержание:\n"
            + "\n".join(content)
        )

    return "\n\n".join(parts)


def generic_to_text(item: dict) -> str:
    return json.dumps(
        item,
        ensure_ascii=False
    )


def item_to_document(item: dict) -> Document:

    item_type = item.get("type")

    if item_type == "theorem":
        text = theorem_to_text(item)

    elif item_type == "definition":
        text = definition_to_text(item)

    else:
        text = generic_to_text(item)

    metadata = {
        "knowledge_id": item.get("id"),
        "type": item_type,
        "chapter": item.get("chapter"),
        "chapter_name": item.get("chapter_name"),
        "section": item.get("section"),
        "source_file": item.get("source", {}).get("file"),
        "start_line": item.get("source", {}).get("start_line"),
        "end_line": item.get("source", {}).get("end_line"),
    }

    return Document(
        page_content=text,
        metadata=metadata,
    )

In [32]:
def load_documents(path: str) -> list[Document]:

    documents = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                item = json.loads(line)

            except json.JSONDecodeError as e:
                print(
                    f"WARNING: invalid JSON "
                    f"at line {line_number}: {e}"
                )
                continue

            documents.append(
                item_to_document(item)
            )

    return documents

In [33]:
documents = load_documents(KNOWLEDGE_FILE)
len(documents)

151

In [34]:
embeddings = FastEmbeddingEmbed(EMBEDDING_MODEL)

/tmp/ipykernel_53666/2110182448.py:5: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  self.model = TextEmbedding(model_name=model_name)
Fetching 5 files: 100%|██████████| 5/5 [00:34<00:00,  6.88s/it]


In [36]:
client = QdrantClient(QDRANT_URL)

In [38]:
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"Collection {COLLECTION_NAME} deleted")


In [39]:
vectorstore = QdrantVectorStore.from_documents(documents, embedding=embeddings, url=QDRANT_URL, collection_name=COLLECTION_NAME)

# RAG

In [49]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

In [40]:
LLM_MODEL = "qwen2.5:7b"
TOP_K = 5

In [42]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

In [43]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

In [51]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Ты — учитель математики.

Отвечай на вопрос, используя ТОЛЬКО предоставленный контекст.

Правила:

1. Не выдумывай теоремы и формулы.
2. Если ответа нет в контексте, честно скажи:
   "В предоставленных конспектах я не нашёл ответа."
3. Когда решаешь задачу, объясняй ход решения последовательно.
4. Формулы сохраняй в LaTeX.
5. Если в контексте есть доказательство подходящей теоремы,
   используй его как основу объяснения.
6. Не утверждай, что в конспекте сказано то, чего там нет.
7. Не добавляй в ответ ничего, связанного с источниками.

Контекст:

{context}
""",
    ),
    (
        "human",
        "{question}",
    ),
])

In [52]:
def format_docs(docs):

    formatted = []

    for i, doc in enumerate(docs, start=1):

        metadata = doc.metadata

        header = (
            f"[Источник {i}]\n"
            f"Тип: {metadata.get('type')}\n"
            f"Раздел: {metadata.get('section')}\n"
            f"Файл: {metadata.get('source_file')}\n"
            f"Строки: "
            f"{metadata.get('start_line')}-"
            f"{metadata.get('end_line')}"
        )

        formatted.append(
            header
            + "\n\n"
            + doc.page_content
        )

    return "\n\n---\n\n".join(
        formatted
    )


In [53]:
def answer(question: str):
    docs = retriever.invoke(question)

    context = format_docs(docs)

    chain = (
        prompt
        | llm
        | StrOutputParser()
    )

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response, docs

In [54]:
a, b = answer("Как найти координату точки C, делящей отрезок AB в отношении λ?")
a

'Для нахождения координаты точки \\( C \\), которая делит направленный отрезок \\( AB \\) в отношении \\( \\lambda \\), можно использовать следующую формулу:\n\n\\[ X_C = \\frac{X_A + \\lambda X_B}{1 + \\lambda} \\]\n\nЭта формула доказана в разделе "Величина направленного отрезка. Теорема Шаля. Декартова система координат на прямой" (источник 3).\n\n### Объяснение:\n1. **Определение**: Точка \\( C \\) делит направленный отрезок \\( AB \\) в отношении \\( \\lambda \\), если \\( \\lambda = \\frac{AC}{CB} \\).\n2. **Формула координаты точки**:\n   - Если точка \\( C \\) лежит между точками \\( A \\) и \\( B \\), то \\( X_C \\) вычисляется по формуле выше.\n   - Если точка \\( C \\) лежит вне отрезка \\( AB \\), то также используется эта же формула, но с учетом того, что знаки координат могут быть разными.\n\nТаким образом, для нахождения координаты точки \\( C \\), достаточно подставить значения \\( X_A \\) и \\( X_B \\) в указанную формулу.'

In [56]:
a, b = answer("Напомни мне уравнение эллипса")
a

'Уравнение эллипса, которое называется каноническим, имеет вид:\n\n\\[\n\\frac{x^2}{a^2} + \\frac{y^2}{b^2} = 1,\n\\]\n\nгде \\(a\\) и \\(b\\) — semi-major и semi-minor axes соответственно. Если \\(b < a\\), то фокусы эллипса лежат на оси \\(Ox\\). В противном случае, если \\(b > a\\), фокусы будут лежать на оси \\(Oy\\).\n\nЭто уравнение описывает эллипс, центрированный в начале координат.'